# '02.-Actualización Contable Marine' - PROGRAM 02

##  Overview
In this program, we use the bd_update.pkl processed in the previous program.
After that, load and prepare the Accounting information.
Cross the information of bd_update and the account information.
Finally, create the formulated variables and export.
This program creates the final database.

##  Execution Flow
1. Library Import.
2. Path Definition and Macrovariables. 
3. Data import (bd_update.pkl & Account Information). 
4. Data Format.
5. Data Cross.
6. Creation of formulated variables.
7. OSLR and Dates validation.
8. Final adjustments.
9. Final Export.

##  Output
- Excel file with the final processed database of the month. ({AñoMes}_Siniestros_Marine.xlsx)
- Pickle file with the final processed database of the month. ({AñoMes}_Siniestros_Marine.pkl)
- Excel file with relevant information for incidences analysis. (actuarial.xlsx)

## 1. Library import

In [100]:
## =============================================================================
# Section 1: Library import
# =============================================================================

# === Optional: Clean all the enviroment prior running === #
%reset -f                                                  
# ======================================================== #
import os
import pandas as pd
#import dtale
import numpy as np
import re
import timeit
import sqlite3

start_time = timeit.default_timer() # Timer

## 1.5 Importación de configuración centralizada

In [101]:
# =============================================================================
# Importación centralizada de configuración
# =============================================================================
import sys
sys.path.insert(0, r'C:/Users/IKAL14/Documents/Integral/Marine/Diccionarios')
from config_marine import *

## 2. Path Definition and Macrovariables 

In [102]:
## =============================================================================
# Section 2: Path Definition and Macrovariables. 
# =============================================================================
print('===============================================================================================')
print('Inicio del proceso')
start_time = timeit.default_timer() # Timer

# ================= Edit variables month================================ #
AñoMes = 202509
AñoMes_ant = 202504

# Convert the variables in datetime 
date_start_AñoMes = pd.to_datetime(str(AñoMes), format='%Y%m') + pd.offsets.MonthBegin(0)
date_start_AñoMes_ant = pd.to_datetime(str(AñoMes_ant), format='%Y%m') + pd.offsets.MonthBegin(0)
date_start_suma = date_start_AñoMes_ant + pd.offsets.MonthBegin(1) # This is an Aux Variable and it should be used when we are proccesing more than 1 month

print(f'Fecha de inicio para considerar en la suma: {date_start_suma}')

# =================Routes definition========================================= #
print('===============================================================================================')

# Define your project directory path as a variable
directorio_proyecto =  "C:/Users/IKAL14/Documents/Integral/Marine"   #C:/Users/KOT23/Documents/Integral

# Change the current working directory to the project directory
os.chdir(directorio_proyecto) 

# Verify that the current directory is the project directory
print(f"Directorio actual de trabajo: {os.getcwd()}")

# Processed files path
ruta_procesados = rf"{directorio_proyecto}/Procesados/{AñoMes}" 
print(f"Ruta de archivos procesados: {ruta_procesados}")

#Ruta de Bases
ruta_bases= rf"{directorio_proyecto}/Bases"
print(f"Ruta de bases: {ruta_bases}")

# Incidences files path
ruta_incidencias = rf"{directorio_proyecto}/Incidencias/{AñoMes}" 
print(f"Ruta de Incidencias: {ruta_incidencias}")

# Validation files path
route_validations = rf"{directorio_proyecto}/Validaciones/{AñoMes}" 
print(f"Ruta de Validaciones: {route_validations}")

# Catalog files path
route_catalog = rf'{directorio_proyecto}/Catalogos'
print(f'Ruta de catalogos: {route_catalog}')

# Actuarial database path
route_actuarial = rf'C:/Users/IKAL14/Documents/Integral/Insumos'
print(f'Ruta de base actuarial: {route_actuarial}')

# Payments information
route_account = fr'C:/Users/IKAL14/Documents/Integral/Insumos/Contabilidad/{AñoMes}'
print(f'Ruta de base de payments: {route_account}')

# 'Pruebas' Route
route_pruebas = rf"{directorio_proyecto}/Procesados/{AñoMes}/pruebas" 
# Verify if the path exists
if not os.path.exists(route_pruebas):
    os.makedirs(route_pruebas)
    print(f"Carpeta creada: {route_pruebas}")
else:
    print(f"La carpeta ya existe: {route_pruebas}")

# 'OSLR' Route
route_oslr = rf"{directorio_proyecto}/Procesados/202504" 

# This DB includes all of the records that can be updated
df_update_db = pd.read_pickle(f'{ruta_procesados}/bd_update.pkl')
df_update_db['CLAIM NUMBER'] = df_update_db['CLAIM NUMBER'].astype(str)
df_update_db['LoB-Inward'] = df_update_db['LoB-Inward'].astype(str)

Inicio del proceso
Fecha de inicio para considerar en la suma: 2025-05-01 00:00:00
Directorio actual de trabajo: C:\Users\IKAL14\Documents\Integral\Marine
Ruta de archivos procesados: C:/Users/IKAL14/Documents/Integral/Marine/Procesados/202509
Ruta de bases: C:/Users/IKAL14/Documents/Integral/Marine/Bases
Ruta de Incidencias: C:/Users/IKAL14/Documents/Integral/Marine/Incidencias/202509
Ruta de Validaciones: C:/Users/IKAL14/Documents/Integral/Marine/Validaciones/202509
Ruta de catalogos: C:/Users/IKAL14/Documents/Integral/Marine/Catalogos
Ruta de base actuarial: C:/Users/IKAL14/Documents/Integral/Insumos
Ruta de base de payments: C:/Users/IKAL14/Documents/Integral/Insumos/Contabilidad/202509
La carpeta ya existe: C:/Users/IKAL14/Documents/Integral/Marine/Procesados/202509/pruebas


## 3. Data import

In [103]:
# =============================================================================
# Section 3: Data import
# =============================================================================

# ======== Import of update database ======== #

# ========== FIX #1: Normalizar CLAIM NUMBER antes de crear KEY LOB ==========
# Elimina espacios internos para garantizar consistencia entre períodos.
df_update_db['CLAIM NUMBER'] = (
    df_update_db['CLAIM NUMBER']
    .str.strip()
    .str.replace(r'\s+', '', regex=True)
)

# ========== FIX #2: Normalizar LoB-Inward antes de crear KEY LOB ==========
# Unifica LoB que cambian de nombre entre períodos para mantener continuidad.
LOB_NORMALIZATION = {
    'CASCO': 'CASCO Y MAQ.',
}
df_update_db['LoB-Inward'] = (
    df_update_db['LoB-Inward']
    .str.strip()
    .replace(LOB_NORMALIZATION)
)

# Crear KEY LOB con formato 100% consistente
df_update_db['KEY LOB'] = df_update_db['CLAIM NUMBER']  + "-" +  df_update_db['LoB-Inward']

# Rename of the columns
df_update_db.rename(columns= {
                              'DEDUCTIBLE': f'DEDUCTIBLE {AñoMes_ant}' , 'DEDUCTIBLE BDX': f'DEDUCTIBLE {AñoMes}',
                              'GROSS RESERVE': f'GROSS RESERVE {AñoMes_ant}', 'GROSS RESERVE BDX': f'GROSS RESERVE {AñoMes}',
                              'Cumulative EXPENSES PAID':f'Cumulative EXPENSES PAID {AñoMes_ant}',
                              'Cumulative VALUATION EXPENSES':f'Cumulative VALUATION EXPENSES {AñoMes_ant}',
                              'Cumulative CLAIMS PAID':f'Cumulative CLAIMS PAID {AñoMes_ant}',
                              'OSLR Inward':f'OSLR Inward {AñoMes_ant}'}, inplace= True)

# ======== Import of account information ======== #
# Initialize a dictionary with the sheets of information
dict_sheets = pd.read_excel(fr'{route_account}/Payments_{AñoMes}.xlsx', sheet_name= None)
# Split the dictionary in dataframes
df_AE = dict_sheets['AE'] 
df_CL = dict_sheets['CL']
df_VA = dict_sheets['VA']

# Cleaning
del dict_sheets

## 4. Data format

In [104]:
# =============================================================================
# Section 4: Data Format
# =============================================================================

# =================== Account information =================== #

# Initialize a list of dataframes, since they have the same structure
list_df_payments = [df_AE,df_CL,df_VA]

# ======= Format Variables ======= #

# === Date Variables === #
cols_date = ['Cover period from', 'Cover period to','Claim Date','Claim Report', 'Date of payment','Payment request date']

# Iterate over the dataframes and columns: strip + convert to datetime (single pass)
for i in range(len(list_df_payments)):
    for col in cols_date:
        list_df_payments[i][col] = list_df_payments[i][col].astype(str).str.strip()
        list_df_payments[i][col] = pd.to_datetime(list_df_payments[i][col], dayfirst=True, errors='coerce')

# Validate that all date columns exist
for i, df in enumerate(list_df_payments):
    missing = [col for col in cols_date if col not in df.columns]
    if missing:
        print(f"❌ DataFrame {i} no tiene las columnas: {missing}")
    else:
        print(f"✅ DataFrame {i} tiene todas las columnas de fecha")

# === String Variables === #
cols_string = ['Claims Reference', 'Lob-Inward']
# Delete spaces; guard against str(NaN) becoming the literal "nan"
for i in range(len(list_df_payments)):
    for col in cols_string:
        list_df_payments[i][col] = list_df_payments[i][col].apply(
            lambda x: str(x).strip().replace(' ', '') if pd.notna(x) and str(x).strip().lower() != 'nan' else x
        )

# Clean 'Policy N°' to avoid mismatches in subsidiary filter
for i in range(len(list_df_payments)):
    list_df_payments[i]['Policy N°'] = list_df_payments[i]['Policy N°'].astype(str).str.strip()

# ======= Data Cleaning ======= #

cols_payment_base = ['Policy N°','Claims Reference','Claim Date','Claim Report','Lob-Inward','Currency','Payment request date','Date of payment','Amount USD (CORRECT)','Status']

# ========== FIX: Mapa canonico de normalizacion de Lob-Inward ==========
lob_normalize_map = {
    'cascoymaquinaria':   'CASCO Y MAQUINARIA',
    'cascoy maquinaria':  'CASCO Y MAQUINARIA',
    'casco y maquinaria': 'CASCO Y MAQUINARIA',   # variante que faltaba (mixed case)
    'casco':              'CASCO Y MAQUINARIA',
    'p&i':                'PANDI',
    'pandi':              'PANDI',
    'dw':                 'DEEP WATERS',
    'deep waters':        'DEEP WATERS',
    'deepwaters':         'DEEP WATERS',
    'cargo':              'CARGO',
    'plataformas':        'PLATAFORMAS',
    'floteles':           'FLOTELES',
}

for i in range(len(list_df_payments)):
    normalized = (
        list_df_payments[i]['Lob-Inward']
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'\s+', ' ', regex=True)   # collapse double spaces
        .map(lob_normalize_map)
    )
    # Only replace where a match was found; preserve original if not in map
    mask_mapped = normalized.notna()
    list_df_payments[i].loc[mask_mapped, 'Lob-Inward'] = normalized[mask_mapped]

# Audit: print unique LoB values post-normalization
print("\n=== Valores unicos de Lob-Inward post-normalizacion ===")
for name, df in [('df_AE', list_df_payments[0]), ('df_CL', list_df_payments[1]), ('df_VA', list_df_payments[2])]:
    print(f"\n{name}:")
    print(df['Lob-Inward'].value_counts(dropna=False).to_string())

# === Create KEY LOB: Claims Reference + Lob-Inward ===
for i in range(len(list_df_payments)):
    list_df_payments[i]['KEY LOB'] = (
        list_df_payments[i]['Claims Reference'].astype(str)
        + "-"
        + list_df_payments[i]['Lob-Inward'].astype(str)
    )

# Update dataframe references
df_AE = list_df_payments[0]
df_CL = list_df_payments[1]
df_VA = list_df_payments[2]

# Verify KEY LOB was created successfully
print("\n✓ Verificando que KEY LOB existe en todos los dataframes:")
for name, df in [('df_AE', df_AE), ('df_CL', df_CL), ('df_VA', df_VA)]:
    if 'KEY LOB' in df.columns:
        print(f"  ✓ {name} contiene KEY LOB")
    else:
        print(f"  ✗ {name} NO contiene KEY LOB")

# Validate: detect "nan" contamination in KEY LOB
print("\n=== Validacion: KEY LOBs con nan (contaminacion por NaN) ===")
for name, df in [('df_AE', df_AE), ('df_CL', df_CL), ('df_VA', df_VA)]:
    bad = df[df['KEY LOB'].str.contains('nan', case=False, na=False)]
    if len(bad) > 0:
        print(f"  ⚠️ {name}: {len(bad)} registros con nan en KEY LOB — revisar Claims Reference vacio")
        print(bad[['Claims Reference', 'Lob-Inward', 'KEY LOB', 'Amount USD (CORRECT)']].head())
    else:
        print(f"  ✅ {name}: sin contaminacion de nan")

# === Filtering by date ===
cols_payment = cols_payment_base + ['KEY LOB']
df_AE_filter = df_AE[(df_AE['Payment request date'] >= date_start_suma) | pd.isna(df_AE['Payment request date'])][cols_payment]
df_CL_filter = df_CL[(df_CL['Payment request date'] >= date_start_suma) | pd.isna(df_CL['Payment request date'])][cols_payment]
df_VA_filter = df_VA[(df_VA['Payment request date'] >= date_start_suma) | pd.isna(df_VA['Payment request date'])][cols_payment]

# NaT audit report
print("\n=== NaT en Payment request date (pasan el filtro por isna) ===")
for name, df in [('AE', df_AE), ('CL', df_CL), ('VA', df_VA)]:
    nat_count = df['Payment request date'].isna().sum()
    print(f"  {name}: {nat_count} NaT de {len(df)} registros ({nat_count/max(len(df),1):.1%})")

# === Filter by Lob-Inward Marine (only canonical values) ===
list_lob_marine = ['CASCO Y MAQUINARIA', 'CARGO', 'PANDI', 'DEEP WATERS', 'PLATAFORMAS', 'FLOTELES']

list_lob_nomarine = []
for df in [df_AE_filter, df_CL_filter, df_VA_filter]:
    valores_fuera = df[~df['Lob-Inward'].isin(list_lob_marine)]['Lob-Inward'].unique()
    list_lob_nomarine.extend(valores_fuera)
list_lob_nomarine = set(list_lob_nomarine)

print(f'\nLoB consideradas para Marine: {list_lob_marine}')
print(f'LoB NO consideradas para Marine: {list_lob_nomarine}')

# Apply marine filter
df_AE_filter = df_AE_filter[df_AE_filter['Lob-Inward'].isin(list_lob_marine)]
df_CL_filter = df_CL_filter[df_CL_filter['Lob-Inward'].isin(list_lob_marine)]
df_VA_filter = df_VA_filter[df_VA_filter['Lob-Inward'].isin(list_lob_marine)]

# === Filter by claims declared in BDX (Codigo 1 / df_update_db) ===
# Solo polizas vigentes (declaradas en el BDX del mes) deben generar
# movimientos financieros en la base final. Pagos de contabilidad sobre
# claims que no estan en el BDX (polizas no vigentes / fuera de cobertura)
# se EXCLUYEN, pero se reportan antes de descartarlos para no perder
# trazabilidad del monto.

# Normalizar CLAIM NUMBER del BDX igual que 'Claims Reference' (sin espacios)
claims_bdx = set(
    df_update_db['CLAIM NUMBER'].astype(str).str.strip().str.replace(' ', '')
)

print(f'\nClaims declarados en BDX (Codigo 1): {len(claims_bdx)}')

list_claims_huerfanos = set()
for nombre, df in [('AE', df_AE_filter), ('CL', df_CL_filter), ('VA', df_VA_filter)]:
    huerfanos = df.loc[~df['Claims Reference'].isin(claims_bdx), 'Claims Reference'].unique()
    if len(huerfanos) > 0:
        list_claims_huerfanos.update(huerfanos)
        monto = df.loc[df['Claims Reference'].isin(huerfanos), 'Amount USD (CORRECT)'].sum()
        print(f'  Warning {nombre}: {len(huerfanos)} claim(s) sin poliza vigente en BDX (${monto:,.2f}): {sorted(huerfanos)}')

print(f'\nTotal claims excluidos por no tener poliza vigente en BDX: {len(list_claims_huerfanos)}')

# Exportar reporte ANTES de descartar, para no perder trazabilidad del monto
if list_claims_huerfanos:
    rows = []
    for nombre, df in [('AE', df_AE_filter), ('CL', df_CL_filter), ('VA', df_VA_filter)]:
        sub = df[df['Claims Reference'].isin(list_claims_huerfanos)].copy()
        if len(sub) > 0:
            sub['ORIGEN'] = nombre
            rows.append(sub)
    df_huerfanos_export = pd.concat(rows, ignore_index=True)
    df_huerfanos_export.to_excel(f'{ruta_procesados}/CLAIMS_SIN_POLIZA_VIGENTE_{AñoMes}.xlsx', index=False)
    print(f'Reporte exportado: CLAIMS_SIN_POLIZA_VIGENTE_{AñoMes}.xlsx')
    print(f'Monto total excluido: ${df_huerfanos_export["Amount USD (CORRECT)"].sum():,.2f}')

# Aplicar el filtro: solo claims con poliza vigente (declarada en BDX)
df_AE_filter = df_AE_filter[df_AE_filter['Claims Reference'].isin(claims_bdx)]
df_CL_filter = df_CL_filter[df_CL_filter['Claims Reference'].isin(claims_bdx)]
df_VA_filter = df_VA_filter[df_VA_filter['Claims Reference'].isin(claims_bdx)]

# === Filter to exclude subsidiary company policies ===
list_subsidiary = ['25300 30028443', '25300 30031974']
subsidiary_clean = [s.strip() for s in list_subsidiary]

df_AE_filter = df_AE_filter[~df_AE_filter['Policy N°'].str.strip().isin(subsidiary_clean)]
df_CL_filter = df_CL_filter[~df_CL_filter['Policy N°'].str.strip().isin(subsidiary_clean)]
df_VA_filter = df_VA_filter[~df_VA_filter['Policy N°'].str.strip().isin(subsidiary_clean)]


# === Group by KEY LOB ===
df_AE_final = df_AE_filter.groupby(by=['KEY LOB', 'Policy N°'])['Amount USD (CORRECT)'].sum().reset_index()
df_AE_final.rename(columns={'Amount USD (CORRECT)': f'Cumulative EXPENSES PAID {AñoMes}'}, inplace=True)

df_CL_final = df_CL_filter.groupby(by=['KEY LOB', 'Policy N°'])['Amount USD (CORRECT)'].sum().reset_index()
df_CL_final.rename(columns={'Amount USD (CORRECT)': f'Cumulative CLAIMS PAID {AñoMes}'}, inplace=True)

df_VA_final = df_VA_filter.groupby(by=['KEY LOB', 'Policy N°'])['Amount USD (CORRECT)'].sum().reset_index()
df_VA_final.rename(columns={'Amount USD (CORRECT)': f'Cumulative VALUATION EXPENSES {AñoMes}'}, inplace=True)


# === Final merge via concat + groupby ===
df_all = pd.concat([df_AE_final, df_CL_final, df_VA_final], ignore_index=True)
df_conta_final = (
    df_all
    .groupby('KEY LOB', as_index=False)
    .agg({
        f'Cumulative EXPENSES PAID {AñoMes}':      'sum',
        f'Cumulative CLAIMS PAID {AñoMes}':        'sum',
        f'Cumulative VALUATION EXPENSES {AñoMes}': 'sum',
        'Policy N°': 'first'
    })
)

# === Reconstruct LoB-Inward on df_conta_final ===
def consolidar_columna(df, base):
    """
    Consolida las variantes de una columna que quedan tras un merge
    (base_old/base_new o base_x/base_y) en una sola columna `base`,
    priorizando la version mas reciente (new/x) y usando la anterior
    (old/y) para rellenar los nulos. Tolera re-ejecuciones fuera de orden:
    si `base` ya existe sin sufijos, no hace nada.
    """
    for suf_new, suf_old in [('_new', '_old'), ('_x', '_y')]:
        col_new, col_old = f'{base}{suf_new}', f'{base}{suf_old}'
        if col_new in df.columns and col_old in df.columns:
            df[base] = df[col_new].fillna(df[col_old])
            df.drop(columns=[col_new, col_old], inplace=True)
        elif col_new in df.columns:
            df.rename(columns={col_new: base}, inplace=True)
        elif col_old in df.columns:
            df.rename(columns={col_old: base}, inplace=True)
    return df

df_conta_final['CLAIM NUMBER'] = df_conta_final['KEY LOB'].str.split('-', n=1).str[0]

df_update_db.columns = df_update_db.columns.str.strip()

for col_base in ['CLAIM NUMBER', 'LoB-Inward']:
    df_update_db = consolidar_columna(df_update_db, col_base)

df_lob_update = (
    df_update_db
    .drop_duplicates(subset='CLAIM NUMBER')
    .set_index('CLAIM NUMBER')['LoB-Inward']
)

df_conta_final['LoB-Inward'] = df_conta_final['CLAIM NUMBER'].map(df_lob_update)

df_lob_all = (
    df_all
    .drop_duplicates(subset='KEY LOB')
    .assign(LoB_Inward=lambda x: x['KEY LOB'].str.split('-', n=1).str[1])
    .set_index('KEY LOB')['LoB_Inward']
)

mask = df_conta_final['LoB-Inward'].isna()
df_conta_final.loc[mask, 'LoB-Inward'] = df_conta_final.loc[mask, 'KEY LOB'].map(df_lob_all)

df_conta_final['KEY LOB'] = (
    df_conta_final['CLAIM NUMBER'].astype(str)
    + "-"
    + df_conta_final['LoB-Inward'].astype(str)
)

# === Final format ===
for col in [f'Cumulative EXPENSES PAID {AñoMes}', f'Cumulative CLAIMS PAID {AñoMes}', f'Cumulative VALUATION EXPENSES {AñoMes}']:
    df_conta_final[col] = df_conta_final[col].fillna(0)

df_conta_final['FLAG CONTA'] = 1

# Print Validation
print(f"\nTenemos un total de ${round(df_conta_final[f'Cumulative EXPENSES PAID {AñoMes}'].sum(), 2)} en Cumulative EXPENSES PAID en el periodo")
print(f"Tenemos un total de ${round(df_conta_final[f'Cumulative CLAIMS PAID {AñoMes}'].sum(), 2)} en Cumulative CLAIMS PAID en el periodo")
print(f"Tenemos un total de ${round(df_conta_final[f'Cumulative VALUATION EXPENSES {AñoMes}'].sum(), 2)} en Cumulative VALUATION EXPENSES en el periodo")

# ===== Handmade Adjustments ===== #
if AñoMes == 202503:
    replacements = {
        '323372640000025-CARGO': '323372640000025-CARGA POLIETILENO',
        '323372640000051-CARGO': '323372640000051-CARGA POLIETILENO'
    }
    df_conta_final['KEY LOB'] = df_conta_final['KEY LOB'].replace(replacements)

cols = ["CLAIM NUMBER", "LoB-Inward", "Policy N°"]
df_conta_final = df_conta_final.drop(columns=cols, errors='ignore')

✅ DataFrame 0 tiene todas las columnas de fecha
✅ DataFrame 1 tiene todas las columnas de fecha
✅ DataFrame 2 tiene todas las columnas de fecha

=== Valores unicos de Lob-Inward post-normalizacion ===

df_AE:
Lob-Inward
ONL                   63
ONPD                  28
NaN                   10
OFFPD                  7
PANDI                  4
CARGO                  3
CASCO Y MAQUINARIA     2
ONLIT                  2
DEEP WATERS            2
OFFL                   1
OEE                    1
PLATAFORMAS            1

df_CL:
Lob-Inward
CARGO                  81
ONPD                   20
EE-T                   20
ONL                    16
NaN                    12
RCTerceros             11
EE                      8
OFFPD                   7
CASCO Y MAQUINARIA      3
PANDI                   3
Remocióndeescombros     2
GastosdeMitigación      1
ONLIT                   1
OEE                     1
Gastosextras            1

df_VA:
Lob-Inward
NaN                   11
ONL                    3
ON

In [105]:
# --- Consolidación de FLAG CONTA tras posibles re-ejecuciones fuera de orden ---
df_update_db = consolidar_columna(df_update_db, 'FLAG CONTA')

print('Consolidación de FLAG CONTA completada. Columnas actuales:')
print([col for col in df_update_db.columns if 'FLAG CONTA' in col])

Consolidación de FLAG CONTA completada. Columnas actuales:
[]


## 5. Data Cross

In [106]:
# =============================================================================
# Section 5: Data Cross
# =============================================================================
print('=' * 95)
print('Comenzamos con el cruce entre la base actualizable del mes y la información contable del mes')
print(f'Tenemos {len(df_update_db)} registros en la base actualizable del mes')
print(f'Tenemos {len(df_conta_final)} registros en la base contable del mes')

# =============================================================================
# PASO 1: Consolidar columnas _x/_y remanentes de re-ejecuciones previas
# =============================================================================
for col_base in ['CLAIM NUMBER', 'LoB-Inward', 'FLAG CONTA']:
    df_update_db = consolidar_columna(df_update_db, col_base)

print(f"\n¿FLAG CONTA existe en df_update_db? {'FLAG CONTA' in df_update_db.columns}")

# =============================================================================
# PASO 2: Preparar y realizar el merge con la información contable
# =============================================================================
cols_extra = [
    'KEY LOB',
    'FLAG CONTA',
    f'Cumulative EXPENSES PAID {AñoMes}',
    f'Cumulative CLAIMS PAID {AñoMes}',
    f'Cumulative VALUATION EXPENSES {AñoMes}'
]
cols_extra_ok = [c for c in cols_extra if c in df_conta_final.columns]
cols_missing = set(cols_extra) - set(cols_extra_ok)
if cols_missing:
    print(f"❌ Columnas faltantes en df_conta_final: {cols_missing}")

registros_antes = len(df_update_db)

df_update_db = df_update_db.merge(
    df_conta_final[cols_extra_ok],
    on='KEY LOB',
    how='outer',  # incluye registros nuevos que solo vienen de contabilidad
    suffixes=('_old', '_new')
)

print(f"\nRegistros antes del merge: {registros_antes}")
print(f"Registros después del merge: {len(df_update_db)}")
print(f"Incremento de registros: {len(df_update_db) - registros_antes}")

# =============================================================================
# PASO 3: Consolidar columnas duplicadas por el merge (_old/_new) y rellenar nulos
# =============================================================================
cols_to_consolidate = [
    'FLAG CONTA',
    f'Cumulative EXPENSES PAID {AñoMes}',
    f'Cumulative CLAIMS PAID {AñoMes}',
    f'Cumulative VALUATION EXPENSES {AñoMes}'
]
for col_base in cols_to_consolidate:
    df_update_db = consolidar_columna(df_update_db, col_base)
    if col_base in df_update_db.columns:
        df_update_db[col_base] = df_update_db[col_base].fillna(0)

if not df_update_db.columns.is_unique:
    print("⚠️ Columnas duplicadas detectadas tras el merge. Corrigiendo...")
    df_update_db = df_update_db.loc[:, ~df_update_db.columns.duplicated()]

# Diagnóstico: mostrar registros con movimientos financieros del mes
nuevos_en_merge = df_update_db[df_update_db['FLAG CONTA'] == 1]
print(f"\nRegistros con movimientos financieros (FLAG CONTA=1): {len(nuevos_en_merge)}")

# =============================================================================
# PASO 4: Evitar doble conteo en KEY LOB con múltiples filas actuariales
# =============================================================================
cols_zerear = [
    f'Cumulative EXPENSES PAID {AñoMes}',
    f'Cumulative VALUATION EXPENSES {AñoMes}'
]

# Criterio de calidad para desempatar entre filas duplicadas del mismo KEY LOB:
# 1) se prefiere la fila con GROSS RESERVE y DEDUCTIBLE no nulos (mas completa)
# 2) entre las completas, se prefiere la de mayor DEDUCTIBLE
# La 'mejor' fila conserva el monto contable; el resto se zerea.
def _mejores_filas_key_lob(df, anio_mes):
    completa = (
        df[f'GROSS RESERVE {anio_mes}'].notna() & df[f'DEDUCTIBLE {anio_mes}'].notna()
    ).astype(int)
    ranking = pd.DataFrame({
        'KEY LOB': df['KEY LOB'],
        '_completa': completa,
        '_deducible': df[f'DEDUCTIBLE {anio_mes}'].fillna(-1),
    }, index=df.index).sort_values(
        by=['KEY LOB', '_completa', '_deducible'], ascending=[True, False, False]
    )
    return ranking.groupby('KEY LOB').head(1).index

mejor_idx_keylob = _mejores_filas_key_lob(df_update_db, AñoMes)
dup_keylob = df_update_db.duplicated(subset=['KEY LOB'], keep=False) & ~df_update_db.index.isin(mejor_idx_keylob)
if dup_keylob.any():
    print(f'\n⚠️  KEY LOBs con múltiples filas actuariales (EXPENSES/VALUATION zerados en duplicados, se conserva la fila mas completa/mayor deducible): {dup_keylob.sum()}')
    for col in cols_zerear:
        if col in df_update_db.columns:
            df_update_db.loc[dup_keylob, col] = 0

# =============================================================================
# PASO 5: Rellenar registros nuevos que solo vienen de contabilidad
# =============================================================================
new_claims_mask = (
    (df_update_db['FLAG CONTA'] == 1) &
    (df_update_db[[f'GROSS RESERVE {AñoMes}', f'DEDUCTIBLE {AñoMes}']].isna().any(axis=1))
)
new_claims_count = new_claims_mask.sum()
print(f"\nRegistros nuevos identificados (solo de contabilidad): {new_claims_count}")

if new_claims_count > 0:
    cols_fill_zero = [
        f'GROSS RESERVE {AñoMes}', f'DEDUCTIBLE {AñoMes}',
        f'GROSS RESERVE {AñoMes_ant}', f'DEDUCTIBLE {AñoMes_ant}',
        f'Cumulative EXPENSES PAID {AñoMes_ant}', f'Cumulative CLAIMS PAID {AñoMes_ant}',
        f'Cumulative VALUATION EXPENSES {AñoMes_ant}', 'Cumulative SALVAGE',
        f'OSLR Inward {AñoMes_ant}'
    ]
    for col in cols_fill_zero:
        if col in df_update_db.columns:
            df_update_db.loc[new_claims_mask, col] = df_update_db.loc[new_claims_mask, col].fillna(0)

    text_defaults = {
        'STATUS': 'P',
        'PAYMENT STATUS': 'OPEN',
        'Monthly': 0,
        'NOTAS': 'Siniestro agregado desde contabilidad'
    }
    for col, default_val in text_defaults.items():
        if col in df_update_db.columns:
            df_update_db.loc[new_claims_mask, col] = df_update_db.loc[new_claims_mask, col].fillna(default_val)

    print("✓ Registros nuevos rellenados correctamente")

# =============================================================================
# PASO 6: Consolidar CLAIM NUMBER y LoB-Inward remanentes del merge
# =============================================================================
for col_base in ['CLAIM NUMBER', 'LoB-Inward']:
    df_update_db = consolidar_columna(df_update_db, col_base)

# =============================================================================
# PASO 7: Validación final
# =============================================================================
print('\n' + '=' * 95)
print('RESULTADO FINAL DEL CRUCE:')
print(f'Registros en df_update_db: {len(df_update_db)}')
columnas_criticas = ['KEY LOB', 'FLAG CONTA', 'CLAIM NUMBER', 'LoB-Inward']
faltantes = [c for c in columnas_criticas if c not in df_update_db.columns]
if faltantes:
    print(f'❌ Columnas críticas faltantes: {faltantes}')
else:
    registros_cruzados = df_update_db['FLAG CONTA'].sum()
    registros_sin_match = (df_update_db['FLAG CONTA'] == 0).sum()
    tasa_match = (registros_cruzados / len(df_update_db)) * 100 if len(df_update_db) > 0 else 0
    print(f'Registros que cruzaron: {registros_cruzados:,.0f}')
    print(f'Registros sin match: {registros_sin_match:,.0f}')
    print(f'Tasa de match: {tasa_match:.2f}%')

Comenzamos con el cruce entre la base actualizable del mes y la información contable del mes
Tenemos 4268 registros en la base actualizable del mes
Tenemos 44 registros en la base contable del mes

¿FLAG CONTA existe en df_update_db? False

Registros antes del merge: 4268
Registros después del merge: 4268
Incremento de registros: 0

Registros con movimientos financieros (FLAG CONTA=1): 47

⚠️  KEY LOBs con múltiples filas actuariales (EXPENSES/VALUATION zerados en duplicados, se conserva la fila mas completa/mayor deducible): 21

Registros nuevos identificados (solo de contabilidad): 1
✓ Registros nuevos rellenados correctamente

RESULTADO FINAL DEL CRUCE:
Registros en df_update_db: 4268
Registros que cruzaron: 47
Registros sin match: 4,221
Tasa de match: 1.10%


## 6. Creation of formulated variables

### Lógica de OSLR (3 pasos)

**1. Cálculo base (sin cambios)**

Se calcula `Cumulative CLAIMS PAID` (gastos + valuación + siniestros pagados, mes
anterior + mes actual) y luego el OSLR inicial:

```
OSLR = max( max(GROSS RESERVE − DEDUCTIBLE, 0) − Cumulative CLAIMS PAID, 0 )
```

⚠️ Excepción: las pólizas en `LIST_LEGACY` se excluyen de este cálculo y quedan intactas.

**2. NUEVO — Validación contra `Net Reserve (USD)` del BDX**

Inmediatamente después del cálculo base y antes de las 6 reglas de zereo.
Aplica solo a los registros **no legacy**, con `STATUS == 'P'` (abiertos) y **solo cuando
`Net Reserve (USD) == 0`**:

- Si `Net Reserve (USD) == 0` y el OSLR calculado no coincide (> tolerancia) → se fuerza el OSLR a 0.
- Si `Net Reserve (USD) != 0` (incluye negativos, ej. líneas CARGO) → el OSLR calculado se conserva
  intacto, para evitar introducir reservas negativas o ajustes no verificados.

Cada ajuste queda auditado en `Ajustes_OSLR_vs_NetReserve.xlsx` (registros afectados y $ de diferencia).

**3. Auditoría — Las 6 reglas de zereo**

Una vez ajustado el OSLR con la validación del BDX, se ejecutan en orden las 6 reglas.
Cada una reporta en consola cuántas filas afecta y cuánto OSLR elimina, y además
cada siniestro tocado queda registrado en `Log_Zereo_OSLR.xlsx` (CLAIM NUMBER, KEY LOB,
STATUS, regla, detalle, OSLR antes/después) — así se puede buscar un siniestro específico
directamente en Excel, sin escribir código, y ver qué regla le movió el OSLR:

1. **Ruido / Redondeo** — el OSLR remanente es insignificante (< `UMBRAL_OSLR_MINIMO`). Se zerea.
2. **Ajuste manual** — el claim está en `OSLR_ZERO_MANUAL` y el mes actual ≤ `periodo_hasta`. Se zerea.
3. **OSLR previo y esperado en cero** — el OSLR del mes pasado fue 0 y el recálculo con variables del mes pasado también da 0. Se zerea (evita "resurrecciones" artificiales).
4. **Legacy nulo** — pólizas de `LIST_LEGACY` con `STATUS == "P"` cuyo OSLR base quedó nulo/vacío. Se fuerza a cero.
5. **Sin cambios en el mes** — el claim está en `list_exceptions_monthly`, `FLAG CONTA == 0`, no es `Monthly`, y ni `GROSS RESERVE` ni `DEDUCTIBLE` variaron vs. el mes pasado. Se zerea el actual y se preserva el OSLR del mes anterior.
6. **KEY LOB duplicado** — el mismo `KEY LOB` aparece en múltiples filas (ej. multi-deducible). Se conserva el OSLR en la fila mas completa (GROSS RESERVE y DEDUCTIBLE no nulos) y, entre las completas, la de mayor DEDUCTIBLE; se zerea en las demás (misma funcion `_mejores_filas_key_lob` usada en PASO 4).

In [107]:
# =============================================================================
# Section 6: Creation of formulated variables
# =============================================================================

# ======= Numerical variables ======= #

# == Exceptions == #
# OSLR
# List of records for wich we use the Gross Reserve and Deductible from previous month
list_ded_reserve = df_update_db[
(df_update_db['NOTAS'] != '')| # All the records that have a commment at this point
(df_update_db['Monthly'] == 1) # All the records that are monthly 
]['CLAIM NUMBER'].unique()

# Initialize the list of claim numbers with incidences during the month
list_exceptions_monthly = df_update_db[
(df_update_db['NOTAS'] != '') & # The claim number that has a comment in 'NOTAS'
(~df_update_db['NOTAS'].isin(['LEGACY INFO', 'LEGACY POLICY'])) # but the comment is different from 'LEGACY INFO', 'LEGACY POLICY'
]['CLAIM NUMBER'].unique()  

# Fill the current values of gross reserve and deductible with the values from previous month 
for col in [f'GROSS RESERVE {AñoMes_ant}', f'DEDUCTIBLE {AñoMes_ant}']:
    target_col = col.replace(str(AñoMes_ant), str(AñoMes))  # Update target column name dynamically
    df_update_db.loc[
        df_update_db['CLAIM NUMBER'].isin(list_ded_reserve),  # If the record is in the list 'list_ded_reserve'
        target_col                                                  # Column to be updated
    ] = df_update_db[col]                                           # Value from the previous month

# Expresión regular para eliminar caracteres ilegales para Excel
def clean_excel_string(s):
    if isinstance(s, str):
        # Quita caracteres no imprimibles ni válidos en XML
        return re.sub(r'[\x00-\x1F\x7F-\x9F]', '', s)
    return s

# Identificar columnas numéricas
numeric_cols = df_update_db.select_dtypes(include=['number']).columns.tolist()
string_cols = df_update_db.select_dtypes(include=['object']).columns.tolist()
print(f"Columnas numéricas: {len(numeric_cols)}")
print(f"Columnas de texto: {len(string_cols)}")

# Aplicar limpieza SOLO a columnas de texto
for col in string_cols:
    df_update_db[col] = df_update_db[col].apply(clean_excel_string)
print("✓ Limpieza completada solo en columnas de texto")

# Debugg
df_update_db.to_excel(f'{ruta_procesados}/pruebas/actuarial.xlsx', index=False)

Columnas numéricas: 16
Columnas de texto: 17


C:\Users\IKAL14\AppData\Local\Temp\ipykernel_176916\4254446606.py:38: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df_update_db.select_dtypes(include=['object']).columns.tolist()


✓ Limpieza completada solo en columnas de texto


In [108]:
# ============== Payments columns ============== #
# == Creation of formulated columns == #
# Agregamos .fillna(0) para que si no hay histórico del mes anterior, se asuma 0 y no destruya la suma.
df_update_db['Cumulative EXPENSES PAID'] = df_update_db[f'Cumulative EXPENSES PAID {AñoMes_ant}'].fillna(0) + df_update_db[f'Cumulative EXPENSES PAID {AñoMes}'].fillna(0)
df_update_db['Cumulative VALUATION EXPENSES'] = df_update_db[f'Cumulative VALUATION EXPENSES {AñoMes_ant}'].fillna(0) + df_update_db[f'Cumulative VALUATION EXPENSES {AñoMes}'].fillna(0)
df_update_db['Cumulative CLAIMS PAID'] = df_update_db[f'Cumulative CLAIMS PAID {AñoMes_ant}'].fillna(0) + df_update_db[f'Cumulative CLAIMS PAID {AñoMes}'].fillna(0)
df_update_db['Total Claims Paid Inward'] = df_update_db['Cumulative EXPENSES PAID'] + df_update_db['Cumulative VALUATION EXPENSES'] + df_update_db['Cumulative CLAIMS PAID'] + df_update_db['Cumulative SALVAGE'].fillna(0)
df_update_db['Total Claims Paid no Alae'] = df_update_db['Total Claims Paid Inward'] - df_update_db['Cumulative EXPENSES PAID']

# ============== OSLR Calculation ============== #
# Paso 1: OSLR = max(max(GR - DED, 0) - Cumulative CLAIMS PAID, 0)
# Paso 2: Validación contra Net Reserve (USD) del BDX 
# Paso 3: 6 reglas de zereo

oslr_col = f'OSLR Inward {AñoMes}'

# 1. Asegurar que todo sea string y sin espacios
df_update_db['INWARD POLICY N°'] = df_update_db['INWARD POLICY N°'].astype(str).str.strip()
LEGACY = [str(p).strip() for p in LIST_LEGACY]

# 2. Hacemos el cálculo matemático primero
calculo_oslr = np.maximum(
    np.maximum(df_update_db[f'GROSS RESERVE {AñoMes}'].fillna(0) - df_update_db[f'DEDUCTIBLE {AñoMes}'].fillna(0), 0)
    - df_update_db['Cumulative CLAIMS PAID'],
    0
)

# 3. Asignamos el resultado SOLO a los que NO son Legacy usando la máscara (.loc)
mask_no_legacy = ~df_update_db['INWARD POLICY N°'].isin(LEGACY)
df_update_db.loc[mask_no_legacy, oslr_col] = calculo_oslr

# =============================================================================
# Paso 2: Validación contra Net Reserve (USD) del BDX original
# Se aplica exclusivamente a los registros NO legacy, con STATUS == 'P' (abiertos)
# y SOLO cuando Net Reserve (USD) == 0. Si Net Reserve != 0 (incluye negativos,
# el OSLR calculado se conserva sin tocar, para evitar introducir reservas negativas.
# Se ejecuta ANTES de las 6 reglas de zereo.
# =============================================================================
TOLERANCIA_NET_RESERVE = 0.10  # tolerancia por redondeo, en USD
net_reserve_col = 'Net Reserve (USD)'
if net_reserve_col not in df_update_db.columns:
    raise KeyError(
        f"No se encontró la columna '{net_reserve_col}' en df_update_db. "
        "Verifica que el BDX incluya esta columna antes de continuar."
    )
df_update_db[net_reserve_col] = df_update_db[net_reserve_col].fillna(0)

# Restringe la validación a NO legacy, STATUS abierto ('P'/'p'), y Net Reserve == 0
mask_validacion_net_reserve = (
    mask_no_legacy
    & df_update_db['STATUS'].isin(['P', 'p'])
    & (df_update_db[net_reserve_col].abs() <= TOLERANCIA_NET_RESERVE)
)

dif_net_reserve = (df_update_db[oslr_col] - df_update_db[net_reserve_col]).abs()
mask_dif_net_reserve = mask_validacion_net_reserve & (dif_net_reserve > TOLERANCIA_NET_RESERVE)

n_dif_net_reserve = int(mask_dif_net_reserve.sum())
monto_ajuste_net_reserve = float(
    (df_update_db.loc[mask_dif_net_reserve, net_reserve_col]
     - df_update_db.loc[mask_dif_net_reserve, oslr_col]).sum()
)

print('\n' + '=' * 95)
print('VALIDACIÓN CONTRA NET RESERVE (USD) DEL BDX (solo Net Reserve == 0)')
print('=' * 95)
print(f"  Registros no-legacy, STATUS='P' y Net Reserve==0 validados: {int(mask_validacion_net_reserve.sum())}")
print(f"  Registros donde OSLR calculado != 0 (se fuerzan a 0): {n_dif_net_reserve}")
print(f"  Ajuste neto aplicado: ${monto_ajuste_net_reserve:,.2f}")

# Log de auditoría: qué se sobreescribió y por cuánto
df_ajustes_net_reserve = df_update_db.loc[mask_dif_net_reserve, [
    'CLAIM NUMBER', 'KEY LOB', oslr_col, net_reserve_col
]].copy()
df_ajustes_net_reserve.rename(columns={
    oslr_col: 'OSLR CALCULADO',
    net_reserve_col: 'NET RESERVE (USD)'
}, inplace=True)
df_ajustes_net_reserve['DIFERENCIA'] = (
    df_ajustes_net_reserve['NET RESERVE (USD)'] - df_ajustes_net_reserve['OSLR CALCULADO']
)
df_ajustes_net_reserve.to_excel(f'{ruta_incidencias}/Ajustes_OSLR_vs_NetReserve.xlsx', index=False)
print(f"  Log exportado: Ajustes_OSLR_vs_NetReserve.xlsx")
print('=' * 95 + '\n')

# Sobreescribir OSLR con el valor exacto de Net Reserve (USD) donde difieren
df_update_db.loc[mask_dif_net_reserve, oslr_col] = df_update_db.loc[mask_dif_net_reserve, net_reserve_col]

# =============================================================================
# LOG DE IMPACTO POR REGLA DE ZEREO
# Cada regla que pone OSLR en 0 reporta cuántos registros y cuánto dinero
# elimina, ANTES de aplicar el zereo — para que el proceso sea auditable.
# =============================================================================
resumen_zereo_oslr = []
log_zereo_oslr_detalle = []  # Detalle por siniestro/regla, para Log_Zereo_OSLR.xlsx

def _zerear_con_log(mask, regla, detalle=''):
    """Zerea oslr_col donde mask es True y registra el impacto (n registros, $ eliminado).
    Ademas guarda una fila por cada siniestro afectado, para poder auditar en Excel
    (sin escribir codigo) que regla movio el OSLR de un siniestro especifico."""
    n = int(mask.sum())
    monto = float(df_update_db.loc[mask, oslr_col].fillna(0).sum())
    resumen_zereo_oslr.append({'REGLA': regla, 'DETALLE': detalle, 'N REGISTROS': n, 'OSLR ELIMINADO': round(monto, 2)})
    print(f"  [{regla}] {n} registro(s) — ${monto:,.2f} de OSLR eliminado. {detalle}")
    if n > 0:
        cols_detalle = [c for c in ['CLAIM NUMBER', 'KEY LOB', 'STATUS','INWARD POLICY N°'] if c in df_update_db.columns]
        detalle_filas = df_update_db.loc[mask, cols_detalle].copy()
        detalle_filas['REGLA'] = regla
        detalle_filas['DETALLE'] = detalle
        detalle_filas['OSLR ANTES'] = df_update_db.loc[mask, oslr_col].fillna(0).values
        detalle_filas['OSLR DESPUES'] = 0.0
        log_zereo_oslr_detalle.append(detalle_filas)
        df_update_db.loc[mask, oslr_col] = 0

print('\n' + '=' * 95)
print('REGLAS DE ZEREO DE OSLR (con impacto en $):')
print('=' * 95)

# Regla 1: OSLR por debajo del umbral mínimo (ruido/redondeo)
UMBRAL_OSLR_MINIMO = 100
OSLR_ZERO_MANUAL = {
    '4431/2023' : 202509,
    '1560/2024' : 202509,
    '1563/2024' : 202509,
    '4478/2024' : 202509,
    '4801/2024' : 202509,
    '4803/2024' : 202509,
    '5704/2024' : 202509,
    '6399/2024' : 202509,
    '6400/2024' : 202509,
    '161/2025'  : 202509,
    '594/2025'  : 202509,
}

mask_umbral = df_update_db[oslr_col] < UMBRAL_OSLR_MINIMO
_zerear_con_log(mask_umbral, 'Umbral mínimo', f'OSLR < {UMBRAL_OSLR_MINIMO}')

# ====================== Handmade Adjustments ====================== #
# Regla 2: Claims zereados manualmente, con vigencia (config_marine.OSLR_ZERO_MANUAL)
def claims_oslr_zero_vigentes(AñoMes):
    """Devuelve los claims cuyo zereo manual de OSLR sigue vigente para AñoMes."""
    return [claim for claim, periodo_hasta in OSLR_ZERO_MANUAL.items() if AñoMes <= periodo_hasta]

claims_zero_vigentes = claims_oslr_zero_vigentes(AñoMes)
mask_manual = df_update_db['CLAIM NUMBER'].isin(claims_zero_vigentes)
_zerear_con_log(mask_manual, 'Ajuste manual', f'{len(claims_zero_vigentes)} claim(s) vigente(s) en config_marine.OSLR_ZERO_MANUAL')

#  ================================================================== #
# Regla 3: OSLR nulo en registros legacy pendientes
mask_legacy_null = (
    df_update_db['INWARD POLICY N°'].isin(LEGACY) & df_update_db['STATUS'].isin(['P', 'p']) & df_update_db[oslr_col].isnull()
)
_zerear_con_log(mask_legacy_null, 'Legacy nulo', 'poliza legacy, STATUS=P, OSLR nulo')

#  ================================================================== #
# Regla 4: KEY LOB duplicado (multi-deducible en el BDX)
# Se conserva el OSLR solo en la fila mas completa/mayor deducible y se zerea en las demas, para evitar que el
# monto contable cruzado por KEY LOB (outer join) infle el OSLR total del claim.
mejor_idx_keylob_oslr = _mejores_filas_key_lob(df_update_db, AñoMes)
mask_dup_keylob_oslr = (
    df_update_db.duplicated(subset=['KEY LOB'], keep=False)
    & ~df_update_db.index.isin(mejor_idx_keylob_oslr)
)
_zerear_con_log(mask_dup_keylob_oslr, 'KEY LOB duplicado', 'se conserva OSLR en la fila con mayor completitud/deducible; se zerea en las demas')

#Regla 5: OSLR 0 en registros vacíos restantes sin importar legacy o status
mask_nulos_restantes = df_update_db[oslr_col].isna()
df_update_db.loc[mask_nulos_restantes, oslr_col] = 0
_zerear_con_log(mask_nulos_restantes, 'Zerar OSLR en registros nulos', 'OSLR nulos cambiados a 0')

# =============================================================================
# Exportar log detallado de zereo de OSLR (por siniestro y regla)
# Este archivo permite buscar un CLAIM NUMBER especifico en Excel y ver que regla(s) le
# modificaron el OSLR.
# =============================================================================
if log_zereo_oslr_detalle:
    df_log_zereo_oslr = pd.concat(log_zereo_oslr_detalle, ignore_index=True)
else:
    df_log_zereo_oslr = pd.DataFrame(columns=['CLAIM NUMBER', 'KEY LOB', 'STATUS', 'REGLA', 'DETALLE', 'OSLR ANTES', 'OSLR DESPUES'])
df_log_zereo_oslr.to_excel(f'{ruta_incidencias}/Log_Zereo_OSLR.xlsx', index=False)
n_claims_unicos = df_log_zereo_oslr['CLAIM NUMBER'].nunique() if len(df_log_zereo_oslr) else 0
print(f"\nLog detallado exportado: Log_Zereo_OSLR.xlsx ({len(df_log_zereo_oslr)} fila(s), {n_claims_unicos} siniestro(s) unico(s))")




# ============== INCURRED ============== #
# Al haber limpiado los NaN de OSLR y de Pagos, estas dos columnas saldrán perfectas y 100% numéricas
df_update_db['Inward Incurred'] = df_update_db['Total Claims Paid Inward'] + df_update_db[oslr_col]
df_update_db['Inward Incurred no Alae'] = df_update_db['Inward Incurred'] - df_update_db['Cumulative EXPENSES PAID']




# ======= Other variables ======= #
df_update_db['YEAR OF THE RESERVE'] = date_start_AñoMes.year
df_update_db['ACCIDENT YEAR'] = df_update_db['DATE OF LOSS'].dt.year
df_update_db['CLAIMS-ID'] = df_update_db['CLAIM NUMBER ORIGINAL'].astype(str) + df_update_db['LoB-Inward BDX'].astype(str)
df_update_db['ATTRITIONAL/LARGE'] = 'ATTRITIONAL'
df_update_db['INWARD POLICY'] = '02-Marine'
df_update_db['StandRe LoB'] = '03-MAT'
df_update_db['StandRe detailed LoB'] = 'Combined marine'
df_update_db['UW Year'] = np.where(
    df_update_db['DATE OF LOSS'].dt.month < 7,
    df_update_db['DATE OF LOSS'].dt.year - 1,
    df_update_db['DATE OF LOSS'].dt.year
)

# Guardar cambios limpios
# df_update_db.to_excel(f'C:/Users/IKAL14/Desktop/CALCULOS.xlsx', index= False)


VALIDACIÓN CONTRA NET RESERVE (USD) DEL BDX (solo Net Reserve == 0)
  Registros no-legacy, STATUS='P' y Net Reserve==0 validados: 1988
  Registros donde OSLR calculado != 0 (se fuerzan a 0): 1108
  Ajuste neto aplicado: $-3,808,739.79
  Log exportado: Ajustes_OSLR_vs_NetReserve.xlsx


REGLAS DE ZEREO DE OSLR (con impacto en $):
  [Umbral mínimo] 3156 registro(s) — $0.02 de OSLR eliminado. OSLR < 100
  [Ajuste manual] 11 registro(s) — $0.00 de OSLR eliminado. 11 claim(s) vigente(s) en config_marine.OSLR_ZERO_MANUAL
  [Legacy nulo] 34 registro(s) — $0.00 de OSLR eliminado. poliza legacy, STATUS=P, OSLR nulo
  [KEY LOB duplicado] 21 registro(s) — $15,943.50 de OSLR eliminado. se conserva OSLR en la fila con mayor completitud/deducible; se zerea en las demas
  [Zerar OSLR en registros nulos] 901 registro(s) — $0.00 de OSLR eliminado. OSLR nulos cambiados a 0

Log detallado exportado: Log_Zereo_OSLR.xlsx (4123 fila(s), 4071 siniestro(s) unico(s))


## 7. OSLR and Dates Validation

In [109]:
# =============================================================================
# Section 7: OSLR and Dates Validation
# =============================================================================
print('\n' + '='*95)
print('SECCIÓN 7: VALIDACIONES DE OSLR Y FECHAS')
print('='*95)

# Identificar registros nuevos (solo de contabilidad) ANTES de hacer validaciones
new_records_mask = (df_update_db['FLAG CONTA'] == 1) & (df_update_db['NOTAS'] == 'Siniestro agregado desde contabilidad')
new_records_count = new_records_mask.sum()

print(f"\n📌 Registros nuevos (excluidos de validaciones estrictas): {new_records_count}")
if new_records_count > 0:
    print(f"   Estos registros se mantienen sin validación de fechas/OSLR")
    new_records_list = df_update_db.loc[new_records_mask, 'KEY LOB'].unique()
    for claim_key in new_records_list:
        print(f"   - {claim_key}")

# ======= Validations ======= #

# == Validation of 'OSLR Inward == #
# DIF OSLR se mantiene como float; la comparación usa tolerancia explícita
# (dif.abs() > 1) en vez de truncar a int
df_update_db['DIF OSLR'] = (df_update_db[f'OSLR Inward {AñoMes}'] - df_update_db[f'OSLR Inward {AñoMes_ant}']).fillna(0)

# Filter the records that meet the following conditions
# EXCLUIR registros nuevos de esta validación
df_errors_oslr = df_update_db[
    (df_update_db['DIF OSLR'].abs() > 1) & # Exist a difference of OSLR within months (tolerancia explícita)
    (df_update_db['FLAG CONTA'] == 0) & # The records doesn't have accounting movements in the month of process
    (df_update_db['Monthly'] == 0) & # The record is not a new claim
    ~(((df_update_db[f'GROSS RESERVE {AñoMes}'] - df_update_db[f'GROSS RESERVE {AñoMes_ant}']) != 0) | #Excludes records that had
     ((df_update_db[f'DEDUCTIBLE {AñoMes}'] - df_update_db[f'DEDUCTIBLE {AñoMes_ant}']) != 0)) &  # a change in Gross Reserve or Deductible from the previous month.
    ~new_records_mask  # EXCLUIR registros nuevos
]

# === Exportar incidencias de OSLR (bug #1: nunca se exportaban) ===
cols_errors_oslr = [
    'KEY LOB',
    f'OSLR Inward {AñoMes_ant}', f'OSLR Inward {AñoMes}', 'DIF OSLR',
    f'GROSS RESERVE {AñoMes}', f'DEDUCTIBLE {AñoMes}', 'FLAG CONTA'
]
cols_errors_oslr = [c for c in cols_errors_oslr if c in df_errors_oslr.columns]
df_errors_oslr[cols_errors_oslr].to_excel(f'{ruta_incidencias}/Incidencias_OSLR.xlsx', index=False)
print(f'Se exportó correctamente el archivo de incidencias de OSLR ({len(df_errors_oslr)} registro(s))')

# == Validation of Dates == #
# EXCLUIR registros nuevos (que no tienen fechas)
df_errors_dates = df_update_db[
    (df_update_db['DATE OF LOSS'] < df_update_db['POLICY PERIOD START DATE']) |
    (df_update_db['DATE OF LOSS'] > df_update_db['POLICY PERIOD END DATE']) |
    ((df_update_db['DATE OF LOSS'] > df_update_db['DATE OF REPORT']) &
    (df_update_db['STATUS'] != 'C')) &
    ~new_records_mask  # EXCLUIR registros nuevos
].copy()

# Sanitize status
df_errors_dates.loc[:, 'STATUS'] = df_errors_dates['STATUS'].astype(str).str.strip().str.capitalize()

# Drop canceled and NaN records from this filtered DataFrame
df_errors_dates = df_errors_dates[~df_errors_dates['STATUS'].isin(['C','Nan',''])]

# Report
print('==============================================================================')
if not df_errors_dates.empty:
    print('Los registros con errores en fechas son los siguientes:')
    for index, row in df_errors_dates.iterrows():
        print(
            f"\nCLAIM NUMBER: {row['CLAIM NUMBER']}\n"
            f"  DATE OF LOSS          : {row['DATE OF LOSS']}\n"
            f"  DATE OF REPORT        : {row['DATE OF REPORT']}\n"
            f"  POLICY PERIOD START   : {row['POLICY PERIOD START DATE']}\n"
            f"  POLICY PERIOD END     : {row['POLICY PERIOD END DATE']}\n"
            f"  STATUS                : {row['STATUS']}\n"
        )
else:
    print("No se encontraron errores en fechas.")
print('==============================================================================')

# Final selection of columns
df_errors_dates = df_errors_dates[['CLAIM NUMBER ORIGINAL','DATE OF LOSS','DATE OF REPORT',
                                   'INWARD POLICY N°','LoB-Inward','POLICY PERIOD START DATE',
                                   'POLICY PERIOD END DATE','STATUS', 'NOTAS']]

# Create the error column according to the type of error
# Define the conditions and its value
conditions = [
    df_errors_dates['DATE OF LOSS'] < df_errors_dates['POLICY PERIOD START DATE'],
    df_errors_dates['DATE OF LOSS'] > df_errors_dates['POLICY PERIOD END DATE'],
    df_errors_dates['DATE OF LOSS'] > df_errors_dates['DATE OF REPORT']
]

# Define the choices
choices = ['DOL antes vigencia', 'DOL después vigencia', 'DOL después reporte']

# Create the column 'ERROR' based on the conditions
df_errors_dates['ERROR'] = np.select(conditions, choices, default='Sin error')
# Exportof the file
df_errors_dates.to_excel(f'{ruta_incidencias}/Incidencias_fechas.xlsx' ,index= False)
print('Se exportó correctamente el archivo de incidencias de fechas')
print('==============================================================================')

# LoB normalization
# Convert the relevant columns to string type
df_update_db[['LoB-Inward BDX', 'LoB-Inward','CLAIM NUMBER ORIGINAL']] = df_update_db[['LoB-Inward BDX', 'LoB-Inward','CLAIM NUMBER ORIGINAL']].astype(str)

# Replace 'nan' strings with empty strings in 'LoB-Inward BDX'
df_update_db['LoB-Inward BDX'] = df_update_db['LoB-Inward BDX'].replace('nan', '')

# Fill empty values in 'LoB-Inward BDX' with the values from 'LoB-Inward'
df_update_db.loc[df_update_db['LoB-Inward BDX'] == '', 'LoB-Inward BDX'] = df_update_db['LoB-Inward']
df_update_db['CLAIMS-ID'] = df_update_db['CLAIM NUMBER ORIGINAL'] + "-" + df_update_db['LoB-Inward BDX']

# Debugg
#df_update_db.to_excel(f'{ruta_procesados}/pruebas/actuarial.xlsx', index= False )


SECCIÓN 7: VALIDACIONES DE OSLR Y FECHAS

📌 Registros nuevos (excluidos de validaciones estrictas): 0
Se exportó correctamente el archivo de incidencias de OSLR (40 registro(s))
Los registros con errores en fechas son los siguientes:

CLAIM NUMBER: 323361640000029
  DATE OF LOSS          : 2023-03-08 00:00:00
  DATE OF REPORT        : 2023-05-30 00:00:00
  POLICY PERIOD START   : 2021-02-20 00:00:00
  POLICY PERIOD END     : 2023-02-20 00:00:00
  STATUS                : P

Se exportó correctamente el archivo de incidencias de fechas


## 7.5 Validación empírica de OSLR (patrón histórico open/closed)

Chequeo adicional de plausibilidad (no reemplaza la fórmula oficial de la Sección 6).
Basado en el análisis estadístico de `202509_Siniestros_Marine_MANUAL.xlsx` (3,921 filas):

- **OSLR > 0** casi siempre (98%) corresponde a un siniestro **aún no pagado**
  (`Total Claims Paid Inward == 0`) y **reportado recientemente** (≤ 730 días).
- **OSLR = 0** domina en siniestros ya pagados y/o antiguos.
- Precisión de la heurística contra los datos reales: **97.7%**.

Marca como "atípicos" los registros cuyo OSLR calculado (Sección 6) no sigue este
patrón histórico, para revisión manual — no implica que estén mal calculados.

In [110]:
# =============================================================================
# Section 7.5: Validación empírica de OSLR (patrón histórico open/closed)
# =============================================================================
# Regla inferida estadísticamente (ver oslr_rules.py):
#   oslr_expected_status = 'open'   si Total Claims Paid Inward == 0
#                                       y días desde reporte <= DIAS_UMBRAL_OSLR_ABIERTO
#                         = 'closed' en cualquier otro caso

print('\n' + '='*95)
print('SECCIÓN 7.5: VALIDACIÓN EMPÍRICA DE OSLR (PATRÓN OPEN/CLOSED)')
print('='*95)

DIAS_UMBRAL_OSLR_ABIERTO = 730
fecha_corte = date_start_AñoMes + pd.offsets.MonthEnd(0)

def oslr_expected_status(paid_inward: float, days_since_report: float,
                          days_threshold: int = DIAS_UMBRAL_OSLR_ABIERTO) -> str:
    """Heurística empírica (no oficial): reproduce el patrón observado en datos históricos."""
    if pd.isna(days_since_report):
        return 'closed'
    if paid_inward == 0 and days_since_report <= days_threshold:
        return 'open'
    return 'closed'

df_check = df_update_db.copy()
df_check['days_since_report'] = (fecha_corte - df_check['DATE OF REPORT']).dt.days
df_check['oslr_actual_status'] = np.where(df_check[oslr_col].fillna(0) > 0, 'open', 'closed')
df_check['oslr_expected_status'] = df_check.apply(
    lambda r: oslr_expected_status(r['Total Claims Paid Inward'], r['days_since_report']),
    axis=1
)

mask_atipicos = df_check['oslr_actual_status'] != df_check['oslr_expected_status']
df_atipicos_oslr = df_check.loc[mask_atipicos, [
    'KEY LOB', 'CLAIM NUMBER', 'DATE OF REPORT', 'days_since_report',
    'Total Claims Paid Inward', oslr_col, 'oslr_actual_status', 'oslr_expected_status'
]].copy()

accuracy = 1 - mask_atipicos.mean()
print(f"Precisión de la heurística vs. OSLR real: {accuracy:.2%}")
print(f"Registros atípicos (no siguen el patrón histórico): {len(df_atipicos_oslr)}")
print("  Nota: no implica error de cálculo — solo marca registros a revisar manualmente")
if not df_check.empty:
    print(pd.crosstab(df_check['oslr_actual_status'], df_check['oslr_expected_status']))

df_atipicos_oslr.to_excel(f'{ruta_incidencias}/Incidencias_OSLR_Patron_Empirico.xlsx', index=False)
print('Se exportó correctamente el archivo de incidencias del patrón empírico de OSLR')
print('='*95 + '\n')


SECCIÓN 7.5: VALIDACIÓN EMPÍRICA DE OSLR (PATRÓN OPEN/CLOSED)
Precisión de la heurística vs. OSLR real: 96.46%
Registros atípicos (no siguen el patrón histórico): 151
  Nota: no implica error de cálculo — solo marca registros a revisar manualmente
oslr_expected_status  closed  open
oslr_actual_status                
closed                  3965   130
open                      21   152
Se exportó correctamente el archivo de incidencias del patrón empírico de OSLR



## 8. Final Adjustments

In [111]:
# =============================================================================
# Section 8: Final adjustments
# =============================================================================

print('\n' + '='*95)
print('SECCIÓN 8: AJUSTES FINALES')
print('='*95)

# Verificar que las columnas necesarias existan ANTES de hacer operaciones
print('\n🔍 Verificando columnas críticas para final adjustment:')

# Rellenar cualquier NaN restante en columnas críticas
critical_cols = [
    f'GROSS RESERVE {AñoMes}', f'DEDUCTIBLE {AñoMes}',
    'LoB-Inward BDX', 'CLAIM NUMBER ORIGINAL',
    'Cumulative SALVAGE', 'Cumulative EXPENSES PAID', 'Cumulative VALUATION EXPENSES'
]

for col in critical_cols:
    if col in df_update_db.columns:
        nulos = df_update_db[col].isna().sum()
        if nulos > 0:
            # Rellenar NaN con 0 para columnas numéricas, vacío para texto
            if df_update_db[col].dtype in ['float64', 'int64']:
                df_update_db[col] = df_update_db[col].fillna(0)
            else:
                df_update_db[col] = df_update_db[col].fillna('')
            print(f"  ✓ {col}: {nulos} valores rellenados")

df_update_db.drop(columns=['LoB-Inward','CLAIM NUMBER'], inplace= True)
df_update_db.rename(columns={'LoB-Inward BDX': 'LoB-Inward','CLAIM NUMBER ORIGINAL' :'CLAIM NUMBER',
                             f'OSLR Inward {AñoMes}': 'OSLR Inward',
                             f'GROSS RESERVE {AñoMes}': f'GROSS RESERVE',
                             f'DEDUCTIBLE {AñoMes}': f'DEDUCTIBLE'
                             }, inplace= True)

# Rename of  policies 
df_update_db.loc[df_update_db['INWARD POLICY N°'] == '90600-323484','INWARD POLICY N°'] = '90600 323484'
df_update_db.loc[df_update_db['INWARD POLICY N°'] == '90600-328256','INWARD POLICY N°'] = '90600 328256'

#LoB normalization
# LoB of legacy info
df_update_db.loc[df_update_db['LoB-Inward'] == 'CASCO', 'LoB-Inward'] = 'CASCO Y MAQUINARIA'

# Subsidiary Normalization
df_update_db['SUBSIDIARY'] = df_update_db['SUBSIDIARY'].str.strip() # Delete all the spaces at the end and beggining

# Normalize the 'SUBSIDIARY' column
df_update_db["SUBSIDIARY"] = (
df_update_db["SUBSIDIARY"]
    #todo en mayusculas
    .str.upper()
    #remueve la palabra pemex
    .str.replace(r"\bPEMEX\b", "", regex=True)
    #remueve espacios extras
    .str.replace(r"\s+", " ", regex=True)
    #elimina espacios al inicio y final
    .str.strip()
)
# Initialize a dictionary with the correct names
dict_subsidiaries = {
    'PEMEX Logistica':'LOGÍSTICA',
    'LOGISTICA':'LOGÍSTICA',
    'Logistica':'LOGÍSTICA',
    'Pemex Logística':'LOGÍSTICA',
    'LOG':'LOGÍSTICA'
}
# Apply the correction 
df_update_db['SUBSIDIARY'] = df_update_db['SUBSIDIARY'].replace(dict_subsidiaries)

#Cambiamos los valores de la columna Subsidiary
df_update_db['SUBSIDIARY'] = df_update_db['SUBSIDIARY'].replace({
'LOGÍSTICA':" DIRECCIÓN DE LOGÍSTICA Y SALVAGUARDIA ESTRATÉGICA ",
'REFINACIÓN':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA",
'PETROQUÍMICA':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA", 
'EXPLORACIÓN Y PRODUCCIÓN':"DIRECCIÓN DE EXPLORACIÓN Y EXTRACCIÓN", 
'PMI':"DIRECCIÓN DE EXPLORACIÓN Y EXTRACCIÓN",
'ETILENO':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA", 
'PERFORACIÓN Y SERVICIOS':"DIRECCIÓN DE EXPLORACIÓN Y EXTRACCIÓN",
'TRANSFORMACIÓN INDUSTRIAL':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA",
'REF':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA",
'PEP':"DIRECCIÓN DE EXPLORACIÓN Y EXTRACCIÓN",
'PTRI':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA",
'GAS Y PETROQUÍMICA BÁSICA':"  DIRECCIÓN DE PROCESOS INDUSTRIALES Y DIRECCIÓN DE TRANSFORMACIÓN ENERGÉTICA"})

# Fill NaN with 'NO ESPECIFICADO'
df_update_db['SUBSIDIARY'] = df_update_db['SUBSIDIARY'].fillna('NO ESPECIFICADO')

#Columns to drop
# Columnas fijas que siempre se eliminan
drop_fixed = [
    'KEY DED', 'CLAIM NUMBER ORIGINAL BDX',
    'POLICY PERIOD START DATE BDX', 'POLICY PERIOD END DATE BDX',
    'Monthly', 'CEDANT OBSERVATIONS', 'KOT OBSERVATIONS', 'DIF OSLR'
]
# Columnas con sufijo de período (YYYYMM) que quedan tras el procesamiento
drop_period = [col for col in df_update_db.columns if re.search(r'\s\d{6}$', col)]
dropcolumns = drop_fixed + drop_period

# Final order
list_columns_order = [
'YEAR OF THE RESERVE','ACCIDENT YEAR','CLAIMS-ID','CLAIM NUMBER','DATE OF LOSS','DATE OF REPORT','ATTRITIONAL/LARGE',	
'INWARD POLICY','INWARD POLICY N°','POLICY PERIOD START DATE','POLICY PERIOD END DATE','SUBSIDIARY','StandRe LoB',
'StandRe detailed LoB','LoB-Inward','COVER','COVERAGE','STATUS','REF. PEMEX','LOCATION','DESCRIPTION' ,
'Cumulative SALVAGE','Cumulative EXPENSES PAID','Cumulative VALUATION EXPENSES','Cumulative CLAIMS PAID',
'Total Claims Paid Inward','Total Claims Paid no Alae','OSLR Inward','Inward Incurred',
'Inward Incurred no Alae',f'GROSS RESERVE',f'DEDUCTIBLE','KOT SHARE','UW Year','Nat Cat', 'KEY LOB','FLAG CONTA','NOTAS'
] #,'DIF OSLR', 'FLAG CONTA'

# Filtrar solo las columnas que existen en el dataframe
list_columns_order_final = [col for col in list_columns_order if col in df_update_db.columns]

# Agregar las columnas restantes que no estén en el orden especificado
cols_faltantes = [col for col in df_update_db.columns if col not in list_columns_order_final]
list_columns_order_final.extend(cols_faltantes)

print(f'\n✓ Reordenando columnas (Orden final: {len(list_columns_order)} columnas)')
df_update_db = df_update_db[list_columns_order]
print(f'✓ Nuestra base final cuenta con {len(df_update_db)} registros')


SECCIÓN 8: AJUSTES FINALES

🔍 Verificando columnas críticas para final adjustment:
  ✓ DEDUCTIBLE 202509: 1 valores rellenados
  ✓ LoB-Inward BDX: 971 valores rellenados

✓ Reordenando columnas (Orden final: 38 columnas)
✓ Nuestra base final cuenta con 4268 registros


In [112]:
# =============================================================================
#  Validación de pagos antes de exportar
# =============================================================================
def validar_pagos_antes_exportacion(
    df_conta_final, df_update_db,
    key_col="KEY LOB", claim_col="CLAIMS-ID"
):
    """
    Valida que todo lo que está en df_conta_final
    exista en df_update_db, tanto por KEY LOB como por CLAIMS-ID.
    Detiene el código si hay faltantes.
    """
    errores = []

    # ── Validación 1: KEY LOB ──────────────────────────────────────────
    keys_conta = set(df_conta_final[key_col])
    keys_db    = set(df_update_db[key_col])
    faltantes_key = keys_conta - keys_db

    if faltantes_key:
        df_f = df_conta_final[
            df_conta_final[key_col].isin(faltantes_key)
        ][[key_col]].drop_duplicates()
        errores.append(
            f"[1] KEY LOB de df_conta_final no encontrados en df_update_db: {len(df_f)}\n"
            + df_f.to_string(index=False)
        )

    # ── Resultado ─────────────────────────────────────────────────────
    if errores:
        sep = "=" * 55
        detalle = ("\n" + "─" * 55 + "\n").join(errores)
        raise ValueError(
            f"\n{sep}\n"
            f"  EXPORTACIÓN DETENIDA — {len(errores)} problema(s)\n"
            f"{sep}\n"
            f"{detalle}\n"
            f"{sep}\n"
            f"  Corrige los errores antes de exportar."
        )

    print("✓ Validación OK — todos los pagos de df_conta_final están en df_update_db.")
# ── paso 6: validar ANTES de exportar ─────────────────
validar_pagos_antes_exportacion(
    df_conta_final = df_conta_final,
    df_update_db   = df_update_db,
)

#Redondear columnas numéricas a 2 decimales
df_update_db = df_update_db.round(2)

✓ Validación OK — todos los pagos de df_conta_final están en df_update_db.


C:\Users\IKAL14\AppData\Local\Temp\ipykernel_176916\755079885.py:50: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df_update_db = df_update_db.round(2)


## 9. Final Export

In [113]:
# =============================================================================
# Section 9: Final Export
# =============================================================================

# Final database after merge
if not os.path.exists(f'{ruta_procesados}/Final'):
    os.makedirs(f'{ruta_procesados}/Final')

df_update_db.to_excel(f'{ruta_procesados}/Final/{AñoMes}_Siniestros_Marine.xlsx', index=False)
df_update_db.to_pickle(f'{ruta_procesados}/Final/{AñoMes}_Siniestros_Marine.pkl')